# 04 — Modeling
Per-country Ridge regression models predicting `energy_per_capita`. Train on data through 2010, test on 2011-2024.
**Key finding:** Demographic model works for IND (R²=0.905) and IDN (R²=0.472); fails for CHN/JPN/DEU/USA due to policy/structural breaks.
**Input:** `model_df.csv`
**Output:** `country_ridge_models.pkl`, model comparison table

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

model_df = pd.read_csv("model_df.csv")
EXCLUDED_COUNTRIES = ["NGA"]
feature_cols = [c for c in model_df.columns if "_lag1" in c]
print("Features:", feature_cols)
print(f"model_df shape: {model_df.shape}")

Features: ['population_total_lag1', 'population_growth_pct_lag1', 'pop_65_plus_pct_lag1', 'pop_working_age_pct_lag1', 'urban_pop_pct_lag1', 'gdp_per_capita_usd_lag1', 'gdp_growth_pct_lag1']
model_df shape: (520, 27)


In [2]:
# Country-specific features based on 10-year correlation analysis
country_features = {
    "BRA": ["gdp_per_capita_usd_lag1", "pop_working_age_pct_lag1", "urban_pop_pct_lag1"],
    "CHN": ["gdp_per_capita_usd_lag1", "pop_working_age_pct_lag1", "population_total_lag1"],
    "DEU": ["pop_working_age_pct_lag1", "pop_65_plus_pct_lag1"],
    "IDN": ["gdp_per_capita_usd_lag1", "urban_pop_pct_lag1", "pop_working_age_pct_lag1"],
    "IND": ["gdp_per_capita_usd_lag1", "pop_working_age_pct_lag1", "population_total_lag1"],
    "JPN": ["pop_working_age_pct_lag1", "pop_65_plus_pct_lag1"],
    "USA": ["gdp_per_capita_usd_lag1", "pop_working_age_pct_lag1"],
}

countries_to_model = ["BRA", "CHN", "DEU", "IDN", "IND", "JPN", "USA"]
target = "energy_per_capita"
split_year = 2010
country_results = {}

In [3]:
for code in countries_to_model:
    features = country_features[code]
    subset = model_df[model_df["country_code"] == code].dropna(
        subset=features + [target]).sort_values("year")
    train = subset[subset["year"] <= split_year]
    test = subset[subset["year"] > split_year]
    if len(train) < 10 or len(test) < 3:
        print(f"{code}: insufficient data, skipping")
        continue

    X_train, y_train = train[features].values, train[target].values
    X_test, y_test = test[features].values, test[target].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000], cv=5)
    ridge.fit(X_train_scaled, y_train)
    y_pred = ridge.predict(X_test_scaled)

    cv_scores = cross_val_score(ridge, X_train_scaled, y_train, cv=5, scoring="r2")
    mae_pct = (mean_absolute_error(y_test, y_pred) / y_test.mean()) * 100
    r2 = r2_score(y_test, y_pred)

    country_results[code] = {
        "model": ridge, "scaler": scaler,
        "features": features, "target": target,
        "r2_test": r2, "mae_pct": mae_pct,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "alpha": ridge.alpha_,
        "y_test": y_test, "y_pred": y_pred,
        "test_years": test["year"].values
    }
    print(f"{code}: Test R²={r2:.3f} | MAE={mae_pct:.1f}% | CV R²={cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

print("\nModels trained.")

BRA: Test R²=-0.897 | MAE=3.5% | CV R²=0.142 ± 0.647
CHN: Test R²=-17.339 | MAE=49.5% | CV R²=-3.016 ± 3.688
DEU: Test R²=-2.492 | MAE=11.6% | CV R²=-3.323 ± 2.267
IDN: Test R²=0.472 | MAE=7.2% | CV R²=0.060 ± 1.041
IND: Test R²=0.888 | MAE=3.0% | CV R²=-0.087 ± 0.901
JPN: Test R²=-29.015 | MAE=23.7% | CV R²=-17.683 ± 14.548
USA: Test R²=-23.837 | MAE=13.9% | CV R²=-3.586 ± 3.976

Models trained.


In [4]:
# Model comparison summary table
summary_rows = []
for code, res in country_results.items():
    summary_rows.append({
        "Country": code,
        "Test R²": round(res["r2_test"], 3),
        "MAE % of mean": round(res["mae_pct"], 1),
        "CV R²": round(res["cv_mean"], 3),
        "CV Std": round(res["cv_std"], 3),
        "Best alpha": res["alpha"],
        "Tier": "Tier 1" if res["r2_test"] > 0.4 else ("Tier 2" if res["cv_mean"] > 0.5 else "Tier 3")
    })
summary_df = pd.DataFrame(summary_rows).sort_values("Test R²", ascending=False)
print(summary_df.to_string(index=False))

# Save model metadata (sklearn objects can't be pickled cleanly in all envs)
# Save metadata separately, models will need to be retrained in forecasting notebook
model_meta = {k: {j: v for j, v in val.items() if j not in ["model", "scaler"]}
              for k, val in country_results.items()}
with open("country_ridge_models.pkl", "wb") as f:
    pickle.dump(model_meta, f)
print("\nModel metadata saved to country_ridge_models.pkl")
print("NOTE: Re-run the training cell in 05_forecasting to get live model objects.")

Country  Test R²  MAE % of mean   CV R²  CV Std  Best alpha   Tier
    IND    0.888            3.0  -0.087   0.901        1.00 Tier 1
    IDN    0.472            7.2   0.060   1.041        0.01 Tier 1
    BRA   -0.897            3.5   0.142   0.647        1.00 Tier 3
    DEU   -2.492           11.6  -3.323   2.267     1000.00 Tier 3
    CHN  -17.339           49.5  -3.016   3.688        1.00 Tier 3
    USA  -23.837           13.9  -3.586   3.976      100.00 Tier 3
    JPN  -29.015           23.7 -17.683  14.548       10.00 Tier 3

Model metadata saved to country_ridge_models.pkl
NOTE: Re-run the training cell in 05_forecasting to get live model objects.
